In [2]:
import pandas as pd

# Rutas exactas que me has dado
ruta_bbdd_1 = '../dataset_features_temperatura.csv'
ruta_bbdd_2 = '../../data/dataset_features_temperatura.csv'

variables_clave = ['doppler_variance_energy', 'dH_dt_mean', 'svd_sigma_ratio']

def analizar_dataset(ruta, nombre):
    try:
        df = pd.read_csv(ruta)
        # Filtramos solo las columnas que nos interesan
        columnas = ['temperature'] + [v for v in variables_clave if v in df.columns]
        
        # Agrupamos y calculamos la media
        analisis = df[columnas].groupby('temperature').mean()
        
        print(f"\n📊 --- ANALIZANDO: {nombre} --- 📊")
        print(analisis.to_string())
        
    except Exception as e:
        print(f"\n❌ Error al cargar {nombre}: {e}")

# Ejecutamos el análisis para las dos
analizar_dataset(ruta_bbdd_1, "BBDD 1 (26.000 casos - La que usábamos antes)")
analizar_dataset(ruta_bbdd_2, "BBDD 2 (32.000 casos - La alternativa)")


📊 --- ANALIZANDO: BBDD 1 (26.000 casos - La que usábamos antes) --- 📊
             doppler_variance_energy  dH_dt_mean  svd_sigma_ratio
temperature                                                      
15.0                     5551.464762    1.119843        29.522197
19.0                     5668.521396    1.145358        29.198023
23.0                     8539.825785    1.725401        25.177749
28.0                     4753.487311    0.961509        31.417853
33.0                     4947.778776    0.998324        30.999475
40.0                     2926.288775    0.590382        38.067776
45.0                     3091.627489    0.625079        37.079187
52.0                     5630.720661    1.137102        29.443816
58.0                     4316.924928    0.872369        33.072704
62.0                     5163.396128    1.044252        30.391921
70.0                     6642.931578    1.340847        27.653414
80.0                     7213.674552    1.455333        26.697217
90.0 

In [3]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# =============================================================================
# 🎯 1. CONFIGURACIÓN: CAMBIA ESTO PARA EXPLORAR OTRAS VARIABLES
# =============================================================================
CARACTERISTICA_A_ESTUDIAR = 'doppler_spread' # <-- Escribe aquí la variable a probar

print(f"🔬 EXPLORANDO CARACTERÍSTICA: {CARACTERISTICA_A_ESTUDIAR.upper()} 🔬\n")
print("=" * 60)

# =============================================================================
# 2. CONECTAR FUNCIONES Y RUTAS
# =============================================================================
ruta_hugo = Path("../../Hugo").resolve()
if str(ruta_hugo) not in sys.path:
    sys.path.insert(0, str(ruta_hugo))

try:
    from channel_estimator import compute_channel_matrix_from_iq_paths
    from channel_features_complete import extract_channel_matrix_features
except ImportError:
    print("❌ ERROR: No se han podido importar las funciones de Hugo.")
    exit()

RUTA_YAML = Path("../../data/Modulator.yaml")

# Función auxiliar para aplicar las 3 normalizaciones a las medias
def aplicar_normalizaciones(df_medias, col_name):
    # Cogemos los valores crudos para escalar su tendencia
    datos = df_medias[[col_name]].values
    
    df_res = df_medias.copy()
    df_res[f'MinMax(0-1)'] = MinMaxScaler().fit_transform(datos)
    df_res[f'Standard(Z)'] = StandardScaler().fit_transform(datos)
    df_res[f'Robust'] = RobustScaler().fit_transform(datos)
    
    # Redondeamos un poco para que la tabla sea legible en consola
    return df_res.round(4)

# =============================================================================
# 3. ANÁLISIS DE LAS BBDD ANTIGUAS (CSV)
# =============================================================================
rutas_antiguas = {
    "BBDD 1 (26k - Antigua)": "../dataset_features_temperatura.csv",
    "BBDD 2 (32k - Alternativa)": "../../data/dataset_features_temperatura.csv"
}

for nombre, ruta in rutas_antiguas.items():
    try:
        df = pd.read_csv(ruta)
        if CARACTERISTICA_A_ESTUDIAR in df.columns:
            # Agrupar por temperatura y sacar la media
            df_medias = df[['temperature', CARACTERISTICA_A_ESTUDIAR]].groupby('temperature').mean()
            
            # Normalizar
            df_final = aplicar_normalizaciones(df_medias, CARACTERISTICA_A_ESTUDIAR)
            
            print(f"\n📊 --- {nombre} --- 📊")
            print(df_final.to_string())
        else:
            print(f"\n⚠️ La variable '{CARACTERISTICA_A_ESTUDIAR}' no existe en {nombre}.")
    except Exception as e:
        print(f"\n❌ Error al cargar {nombre}: {e}")

# =============================================================================
# 4. ANÁLISIS DE LAS BBDD NUEVAS EN VIVO (.BIN)
# =============================================================================
materiales = ['carton', 'cristal', 'plastico']
condiciones = {'frio': 15.0, 'templado': 40.0, 'caliente': 90.0}
muestras = ['1', '2'] # Suponiendo que hay muestra 1 y 2 por cada vaso

print("\n\n" + "=" * 60)
print("📡 PROCESANDO GRABACIONES EN VIVO (CARTÓN, CRISTAL Y PLÁSTICO)...")
print("=" * 60)

for material in materiales:
    datos_material = []
    ruta_base = Path(f"../../data/{material}")
    
    for cond_str, temp_val in condiciones.items():
        for m in muestras:
            tx_path = ruta_base / cond_str / f"iq_tx_{m}.bin"
            rx_path = ruta_base / cond_str / f"iq_rx_{m}.bin"
            
            if not tx_path.exists() or not rx_path.exists():
                continue
                
            try:
                H = compute_channel_matrix_from_iq_paths(
                    tx_path=tx_path, rx_path=rx_path, yaml_path=RUTA_YAML,
                    fftshift=False, normalize_fft=False, trim_to_complete_frames=True,
                    output_order="mk", verbose=False
                )
                features = extract_channel_matrix_features(H, input_order="mk")
                
                if CARACTERISTICA_A_ESTUDIAR in features:
                    datos_material.append({
                        'temperature': temp_val,
                        CARACTERISTICA_A_ESTUDIAR: features[CARACTERISTICA_A_ESTUDIAR]
                    })
            except Exception as e:
                pass # Ignoramos errores de lectura individuales para no manchar la consola

    if len(datos_material) > 0:
        df_mat = pd.DataFrame(datos_material)
        df_medias_mat = df_mat.groupby('temperature').mean()
        df_final_mat = aplicar_normalizaciones(df_medias_mat, CARACTERISTICA_A_ESTUDIAR)
        
        print(f"\n🧪 --- MATERIAL: {material.upper()} --- 🧪")
        print(df_final_mat.to_string())
    else:
        print(f"\n⚠️ No se encontraron datos o falló la extracción para el material: {material}")

🔬 EXPLORANDO CARACTERÍSTICA: DOPPLER_SPREAD 🔬


📊 --- BBDD 1 (26k - Antigua) --- 📊
             doppler_spread  MinMax(0-1)  Standard(Z)  Robust
temperature                                                  
15.0                 0.2899       0.1685      -0.9520 -0.6735
19.0                 0.2901       0.6024       0.3516  0.1601
23.0                 0.2900       0.3703      -0.3457 -0.2858
28.0                 0.2903       0.8847       1.1999  0.7026
33.0                 0.2900       0.3366      -0.4469 -0.3505
40.0                 0.2898       0.0258      -1.3809 -0.9478
45.0                 0.2904       1.0000       1.5464  0.9242
52.0                 0.2901       0.5190       0.1012  0.0000
58.0                 0.2902       0.6889       0.6117  0.3265
62.0                 0.2903       0.9510       1.3993  0.8301
70.0                 0.2899       0.1183      -1.1029 -0.7700
80.0                 0.2898       0.0000      -1.4584 -0.9973
90.0                 0.2902       0.6440       0.

In [ ]:
.